# ***SetUp***


In [ ]:
from google.colab import drive, userdata
import os, json, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datetime import datetime

drive.mount('/content/drive')

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✓ HF Token set")
except:
    print("⚠ No HF Token")

BASE    = "/content/drive/MyDrive/rare_disease_project"
DATA    = f"{BASE}/data"
RESULTS = f"{BASE}/results"
MODELS  = f"{BASE}/models"

os.makedirs(RESULTS, exist_ok=True)
os.makedirs(MODELS,  exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load full splits
with open(f"{DATA}/splits/full_train.pkl", "rb") as f:
    full_train = pickle.load(f)
with open(f"{DATA}/splits/full_test.pkl", "rb") as f:
    full_test = pickle.load(f)
with open(f"{DATA}/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Remap labels 0..N
all_labels    = sorted(full_train['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

# Apply remap
train_remapped = full_train.copy()
train_remapped['label'] = train_remapped['label'].map(label_remap)

test_filtered  = full_test[
    full_test['label'].isin(all_labels)
].reset_index(drop=True)
test_remapped  = test_filtered.copy()
test_remapped['label'] = test_remapped['label'].map(label_remap)

print(f"✓ Setup complete")
print(f"  Device      : {torch.cuda.get_device_name(0)}")
print(f"  Train size  : {len(full_train)}")
print(f"  Test size   : {len(full_test)}")
print(f"  Num classes : {NUM_CLASSES}")

# **NLP DataLoader**

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2"
)

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts      = df['symptom_text'].tolist()
        self.labels     = df['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(
                                self.labels[idx], dtype=torch.long)
        }

nlp_train_ds = SymptomDataset(train_remapped, tokenizer)
nlp_test_ds  = SymptomDataset(test_remapped,  tokenizer)

nlp_train_loader = DataLoader(nlp_train_ds, batch_size=32,
                               shuffle=True,  num_workers=2)
nlp_test_loader  = DataLoader(nlp_test_ds,  batch_size=32,
                               shuffle=False, num_workers=2)

print(f"✓ NLP DataLoaders ready")
print(f"  Train samples : {len(nlp_train_ds)}")
print(f"  Test samples  : {len(nlp_test_ds)}")
print(f"  Train batches : {len(nlp_train_loader)}")
print(f"  Classes       : {NUM_CLASSES}")

# **Train NLP (Full Data)**

In [ ]:
from transformers import AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

class BioBERTClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "dmis-lab/biobert-base-cased-v1.2"
        )
        hidden = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

NLP_EPOCHS = 10
nlp_model  = BioBERTClassifier(NUM_CLASSES).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = AdamW(nlp_model.parameters(),
                   lr=2e-5, weight_decay=0.01)

total_steps = len(nlp_train_loader) * NLP_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

nlp_train_losses = []
nlp_train_accs   = []

print(f"Training NLP — {NLP_EPOCHS} epochs on {len(nlp_train_ds)} samples")
print(f"Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(NLP_EPOCHS):
    nlp_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in nlp_train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = nlp_model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            nlp_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(nlp_train_loader)
    acc      = correct / total * 100
    nlp_train_losses.append(avg_loss)
    nlp_train_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{NLP_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ NLP training complete")

# **Evaluate NLP**

In [ ]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load full dataset
df = pd.read_csv(f"{DATA}/clean_multimodal_samples.csv")
print(f"Full dataset: {df.shape}")

# Encode labels directly from disease_name
new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

print(f"Unique classes : {df['label'].nunique()}")
print(f"Label range    : {df['label'].min()} → {df['label'].max()}")

# Add symptom_text if missing
if 'symptom_text' not in df.columns:
    def make_symptom_text(sym_str):
        try:
            syms = ast.literal_eval(sym_str)
            return ' [SEP] '.join([s.lower().strip() for s in syms])
        except:
            return str(sym_str)
    df['symptom_text'] = df['symptoms'].apply(make_symptom_text)
    print("✓ symptom_text created")
else:
    print("✓ symptom_text already exists")

# Keep classes with >= 2 samples for stratification
label_counts  = df['label'].value_counts()
valid_labels  = label_counts[label_counts >= 2].index
df_filtered   = df[df['label'].isin(valid_labels)].reset_index(drop=True)

print(f"\nFiltered dataset : {len(df_filtered)} samples")
print(f"Valid classes    : {df_filtered['label'].nunique()}")

# 80/20 split
train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Remap labels 0..N
all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

print(f"\n✓ Splits ready")
print(f"  Train   : {len(train_df)} samples")
print(f"  Test    : {len(test_df)} samples")
print(f"  Classes : {NUM_CLASSES}")
print(f"\n  Sample train labels: {train_df['label'].head().tolist()}")
print(f"  Sample test labels : {test_df['label'].head().tolist()}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn
import torch

# ── Tokenizer ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2"
)

# ── Dataset ────────────────────────────────────────────────────
class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts      = df['symptom_text'].tolist()
        self.labels     = df['label'].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(
                                self.labels[idx], dtype=torch.long)
        }

nlp_train_ds = SymptomDataset(train_df, tokenizer)
nlp_test_ds  = SymptomDataset(test_df,  tokenizer)

nlp_train_loader = DataLoader(nlp_train_ds, batch_size=32,
                               shuffle=True,  num_workers=2)
nlp_test_loader  = DataLoader(nlp_test_ds,  batch_size=32,
                               shuffle=False, num_workers=2)

print(f"✓ DataLoaders ready")
print(f"  Train samples : {len(nlp_train_ds)}")
print(f"  Test samples  : {len(nlp_test_ds)}")
print(f"  Train batches : {len(nlp_train_loader)}")
print(f"  Test batches  : {len(nlp_test_loader)}")
print(f"  Classes       : {NUM_CLASSES}")

# ── Model ──────────────────────────────────────────────────────
class BioBERTClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "dmis-lab/biobert-base-cased-v1.2"
        )
        hidden = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

nlp_model   = BioBERTClassifier(NUM_CLASSES).to(device)
criterion   = nn.CrossEntropyLoss()
NLP_EPOCHS  = 10
optimizer   = AdamW(nlp_model.parameters(),
                    lr=2e-5, weight_decay=0.01)
total_steps = len(nlp_train_loader) * NLP_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print(f"\n✓ Model ready")
print(f"  Parameters : {sum(p.numel() for p in nlp_model.parameters()):,}")

# ── Training ───────────────────────────────────────────────────
nlp_train_losses = []
nlp_train_accs   = []

print(f"\nTraining NLP — {NLP_EPOCHS} epochs")
print("-" * 55)

for epoch in range(NLP_EPOCHS):
    nlp_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in nlp_train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        logits = nlp_model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            nlp_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(nlp_train_loader)
    acc      = correct / total * 100
    nlp_train_losses.append(avg_loss)
    nlp_train_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{NLP_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ Training complete")

# ── Evaluation ─────────────────────────────────────────────────
def evaluate_topk(model, loader, device, is_cnn=False, k=5):
    model.eval()
    all_preds, all_labels = [], []
    top5_correct, total   = 0, 0

    with torch.no_grad():
        for batch in loader:
            if is_cnn:
                inputs = batch['image'].to(device)
                labels = batch['label'].to(device)
                logits = model(inputs)
            else:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['label'].to(device)
                logits         = model(input_ids, attention_mask)

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            topk = logits.topk(k, dim=1).indices
            for i, lbl in enumerate(labels):
                if lbl in topk[i]:
                    top5_correct += 1
            total += labels.size(0)

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels, all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "f1_weighted"  : round(f1_score(
                            all_labels, all_preds,
                            average='weighted',
                            zero_division=0)*100, 2),
        "top5_accuracy": round(top5_correct/total*100, 2),
        "total_samples": total
    }

print("\nEvaluating...")
nlp_metrics = evaluate_topk(nlp_model, nlp_test_loader, device)

print("\n" + "=" * 55)
print("EXP 3 — NLP UPPER BOUND (Full Dataset)")
print("=" * 55)
print(f"  Accuracy     : {nlp_metrics['accuracy']}%")
print(f"  F1 Macro     : {nlp_metrics['f1_macro']}%")
print(f"  F1 Weighted  : {nlp_metrics['f1_weighted']}%")
print(f"  Top-5 Acc    : {nlp_metrics['top5_accuracy']}%")
print(f"  Test samples : {nlp_metrics['total_samples']}")

# Save
torch.save({
    'model_state_dict': nlp_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : nlp_metrics
}, f"{MODELS}/exp3_nlp_full.pt")
print(f"\n✓ NLP model saved")

In [ ]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle, os

BASE    = "/content/drive/MyDrive/rare_disease_project"
DATA    = f"{BASE}/data"
RESULTS = f"{BASE}/results"
MODELS  = f"{BASE}/models"

# Reload full dataset
df = pd.read_csv(f"{DATA}/clean_multimodal_samples.csv")

# Encode labels
new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

# Add symptom_text
if 'symptom_text' not in df.columns:
    def make_symptom_text(sym_str):
        try:
            syms = ast.literal_eval(sym_str)
            return ' [SEP] '.join([s.lower().strip() for s in syms])
        except:
            return str(sym_str)
    df['symptom_text'] = df['symptoms'].apply(make_symptom_text)

# Filter + split
label_counts = df['label'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df_filtered  = df[df['label'].isin(valid_labels)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Remap labels
all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

print(f"✓ Data restored")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

# ── Dataset ────────────────────────────────────────────────────
class ZebraMapImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.samples   = []
        self.transform = transform

        for _, row in df.iterrows():
            try:
                images = row['images']
                if not isinstance(images, list):
                    import ast
                    images = ast.literal_eval(images)
                for img_info in images:
                    path = img_info['path']
                    if os.path.exists(path):
                        self.samples.append({
                            'path' : path,
                            'label': row['label']
                        })
                        break
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            img = Image.open(sample['path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return {
            'image': img,
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Building CNN datasets...")
cnn_train_ds = ZebraMapImageDataset(train_df, train_transform)
cnn_test_ds  = ZebraMapImageDataset(test_df,  test_transform)

cnn_train_loader = DataLoader(cnn_train_ds, batch_size=32,
                               shuffle=True,  num_workers=2)
cnn_test_loader  = DataLoader(cnn_test_ds,  batch_size=32,
                               shuffle=False, num_workers=2)

print(f"✓ CNN DataLoaders ready")
print(f"  Train images  : {len(cnn_train_ds)}")
print(f"  Test images   : {len(cnn_test_ds)}")
print(f"  Train batches : {len(cnn_train_loader)}")

# ── Model ──────────────────────────────────────────────────────
class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1
        )
        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False
        self.features   = nn.Sequential(
            *list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

CNN_EPOCHS = 15
cnn_model  = ResNet50Classifier(NUM_CLASSES).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = AdamW(
    filter(lambda p: p.requires_grad, cnn_model.parameters()),
    lr=1e-4, weight_decay=0.01
)
scheduler  = CosineAnnealingLR(
    optimizer, T_max=CNN_EPOCHS, eta_min=1e-6)

cnn_train_losses = []
cnn_train_accs   = []

print(f"\nTraining CNN — {CNN_EPOCHS} epochs")
print(f"  Images  : {len(cnn_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in cnn_train_loader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = cnn_model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            cnn_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler.step()
    avg_loss = total_loss / len(cnn_train_loader)
    acc      = correct / total * 100
    cnn_train_losses.append(avg_loss)
    cnn_train_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{CNN_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ CNN training complete")

# ── Evaluation ─────────────────────────────────────────────────
print("\nEvaluating CNN...")
cnn_metrics = evaluate_topk(
    cnn_model, cnn_test_loader, device, is_cnn=True)

print("\n" + "=" * 55)
print("EXP 3 — CNN UPPER BOUND (Full Dataset)")
print("=" * 55)
print(f"  Accuracy     : {cnn_metrics['accuracy']}%")
print(f"  F1 Macro     : {cnn_metrics['f1_macro']}%")
print(f"  F1 Weighted  : {cnn_metrics['f1_weighted']}%")
print(f"  Top-5 Acc    : {cnn_metrics['top5_accuracy']}%")
print(f"  Test samples : {cnn_metrics['total_samples']}")

# Save
torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : cnn_metrics
}, f"{MODELS}/exp3_cnn_full.pt")
print(f"✓ CNN model saved")

# ── Final Summary ──────────────────────────────────────────────
print("\n" + "=" * 55)
print("DAY 5 COMPLETE ✓")
print("=" * 55)
print(f"  NLP Full — Accuracy : {nlp_metrics['accuracy']}%")
print(f"  NLP Full — Top-5    : {nlp_metrics['top5_accuracy']}%")
print(f"  CNN Full — Accuracy : {cnn_metrics['accuracy']}%")
print(f"  CNN Full — Top-5    : {cnn_metrics['top5_accuracy']}%")
print(f"\n  These are your UPPER BOUND ceiling numbers")
print(f"  Next → Day 6: HAM10000 on Kaggle")

In [ ]:
from google.colab import drive
import json, pickle, os, torch

drive.mount('/content/drive')

BASE   = "/content/drive/MyDrive/rare_disease_project"
MODELS = f"{BASE}/models"
RESULTS = f"{BASE}/results"

# ── Check 1: Model file exists ──────────────────────────────────
model_path = f"{MODELS}/exp3_nlp_full.pt"
if os.path.exists(model_path):
    size = os.path.getsize(model_path) / (1024*1024)
    print(f"✅ Model file exists: exp3_nlp_full.pt ({size:.1f} MB)")
else:
    print(f"❌ Model file NOT found: {model_path}")

# ── Check 2: Load and verify metrics ───────────────────────────
try:
    checkpoint = torch.load(model_path, map_location='cpu')
    print(f"\n✅ Checkpoint loaded successfully")
    print(f"   Keys in checkpoint: {list(checkpoint.keys())}")
    print(f"\n   Metrics:")
    for k, v in checkpoint['metrics'].items():
        print(f"     {k}: {v}")
    print(f"\n   Num classes : {checkpoint['num_classes']}")
except Exception as e:
    print(f"❌ Error loading checkpoint: {e}")

# ── Check 3: Experiment tracker ────────────────────────────────
tracker_path = f"{RESULTS}/experiment_tracker.json"
if os.path.exists(tracker_path):
    with open(tracker_path) as f:
        tracker = json.load(f)
    exp3 = tracker['experiments'].get('exp3_full_upperbound', {})
    print(f"\n✅ Tracker found")
    print(f"   Exp3 status     : {exp3.get('status', 'not set')}")
    print(f"   Exp3 nlp_metrics: {exp3.get('nlp_metrics', 'not saved')}")
else:
    print(f"\n❌ Tracker not found")

Mounted at /content/drive
✅ Model file exists: exp3_nlp_full.pt (417.2 MB)
❌ Error loading checkpoint: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if y

In [ ]:
import torch
import numpy as np

# Fix for PyTorch 2.6 — add safe globals
torch.serialization.add_safe_globals([np.ndarray])

# Load with weights_only=False (safe since it's your own file)
checkpoint = torch.load(model_path, map_location='cpu',
                        weights_only=False)

print(f"✅ Checkpoint loaded successfully")
print(f"\n   Metrics saved inside model:")
for k, v in checkpoint['metrics'].items():
    print(f"     {k}: {v}")
print(f"\n   Num classes : {checkpoint['num_classes']}")

# Now update the tracker with the saved metrics
nlp_metrics = checkpoint['metrics']

with open(f"{RESULTS}/experiment_tracker.json", "r") as f:
    tracker = json.load(f)

tracker['experiments']['exp3_full_upperbound']['status']      = "nlp_complete"
tracker['experiments']['exp3_full_upperbound']['nlp_metrics'] = nlp_metrics

with open(f"{RESULTS}/experiment_tracker.json", "w") as f:
    json.dump(tracker, f, indent=2)

print(f"\n✅ Tracker updated with NLP metrics")
print(f"   Accuracy  : {nlp_metrics['accuracy']}%")
print(f"   Top-5 Acc : {nlp_metrics['top5_accuracy']}%")

✅ Checkpoint loaded successfully

   Metrics saved inside model:
     accuracy: 16.74
     f1_macro: 2.71
     f1_weighted: 10.99
     top5_accuracy: 35.81
     total_samples: 7263

   Num classes : 1199

✅ Tracker updated with NLP metrics
   Accuracy  : 16.74%
   Top-5 Acc : 35.81%


In [ ]:
from google.colab import drive, userdata
import pandas as pd
import ast, os, json, pickle
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from datetime import datetime

drive.mount('/content/drive')

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✓ HF Token set")
except:
    print("⚠ No HF Token")

BASE    = "/content/drive/MyDrive/rare_disease_project"
DATA    = f"{BASE}/data"
RESULTS = f"{BASE}/results"
MODELS  = f"{BASE}/models"

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")

# Restore train_df and test_df
df = pd.read_csv(f"{DATA}/clean_multimodal_samples.csv")

new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

if 'symptom_text' not in df.columns:
    def make_symptom_text(sym_str):
        try:
            syms = ast.literal_eval(sym_str)
            return ' [SEP] '.join([s.lower().strip() for s in syms])
        except:
            return str(sym_str)
    df['symptom_text'] = df['symptoms'].apply(make_symptom_text)

label_counts = df['label'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
df_filtered  = df[df['label'].isin(valid_labels)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

print(f"✓ Data restored")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")

# Load saved NLP metrics
torch.serialization.add_safe_globals([np.ndarray])
checkpoint  = torch.load(f"{MODELS}/exp3_nlp_full.pt",
                         map_location='cpu', weights_only=False)
nlp_metrics = checkpoint['metrics']
print(f"\n✓ NLP metrics restored")
print(f"  Accuracy  : {nlp_metrics['accuracy']}%")
print(f"  Top-5 Acc : {nlp_metrics['top5_accuracy']}%")

Mounted at /content/drive
✓ HF Token set
✓ Device: cuda
✓ Data restored
  Train   : 29049
  Test    : 7263
  Classes : 1199

✓ NLP metrics restored
  Accuracy  : 16.74%
  Top-5 Acc : 35.81%


In [ ]:
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
import matplotlib.pyplot as plt

# ── Dataset ────────────────────────────────────────────────────
class ZebraMapImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.samples   = []
        self.transform = transform

        for _, row in df.iterrows():
            try:
                images = row['images']
                if not isinstance(images, list):
                    images = ast.literal_eval(images)
                for img_info in images:
                    path = img_info['path']
                    if os.path.exists(path):
                        self.samples.append({
                            'path' : path,
                            'label': row['label']
                        })
                        break
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            img = Image.open(sample['path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return {
            'image': img,
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Building CNN datasets...")
cnn_train_ds = ZebraMapImageDataset(train_df, train_transform)
cnn_test_ds  = ZebraMapImageDataset(test_df,  test_transform)

cnn_train_loader = DataLoader(cnn_train_ds, batch_size=32,
                               shuffle=True,  num_workers=2)
cnn_test_loader  = DataLoader(cnn_test_ds,  batch_size=32,
                               shuffle=False, num_workers=2)

print(f"✓ CNN DataLoaders ready")
print(f"  Train images  : {len(cnn_train_ds)}")
print(f"  Test images   : {len(cnn_test_ds)}")
print(f"  Train batches : {len(cnn_train_loader)}")

# ── Model ──────────────────────────────────────────────────────
class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1
        )
        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False
        self.features   = nn.Sequential(
            *list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

CNN_EPOCHS = 15
cnn_model  = ResNet50Classifier(NUM_CLASSES).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = AdamW(
    filter(lambda p: p.requires_grad, cnn_model.parameters()),
    lr=1e-4, weight_decay=0.01
)
scheduler  = CosineAnnealingLR(
    optimizer, T_max=CNN_EPOCHS, eta_min=1e-6)

cnn_train_losses = []
cnn_train_accs   = []

print(f"\nTraining CNN — {CNN_EPOCHS} epochs")
print(f"  Images  : {len(cnn_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in cnn_train_loader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = cnn_model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            cnn_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler.step()
    avg_loss = total_loss / len(cnn_train_loader)
    acc      = correct / total * 100
    cnn_train_losses.append(avg_loss)
    cnn_train_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{CNN_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ CNN training complete")

# ── Evaluation ─────────────────────────────────────────────────
def evaluate_topk(model, loader, device, is_cnn=False, k=5):
    model.eval()
    all_preds, all_labels = [], []
    top5_correct, total   = 0, 0

    with torch.no_grad():
        for batch in loader:
            if is_cnn:
                inputs = batch['image'].to(device)
                labels = batch['label'].to(device)
                logits = model(inputs)
            else:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels         = batch['label'].to(device)
                logits         = model(input_ids, attention_mask)

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            topk = logits.topk(k, dim=1).indices
            for i, lbl in enumerate(labels):
                if lbl in topk[i]:
                    top5_correct += 1
            total += labels.size(0)

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels, all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "f1_weighted"  : round(f1_score(
                            all_labels, all_preds,
                            average='weighted',
                            zero_division=0)*100, 2),
        "top5_accuracy": round(top5_correct/total*100, 2),
        "total_samples": total
    }

print("\nEvaluating CNN...")
cnn_metrics = evaluate_topk(
    cnn_model, cnn_test_loader, device, is_cnn=True)

print("\n" + "=" * 55)
print("EXP 3 — CNN UPPER BOUND (Full Dataset)")
print("=" * 55)
print(f"  Accuracy     : {cnn_metrics['accuracy']}%")
print(f"  F1 Macro     : {cnn_metrics['f1_macro']}%")
print(f"  F1 Weighted  : {cnn_metrics['f1_weighted']}%")
print(f"  Top-5 Acc    : {cnn_metrics['top5_accuracy']}%")
print(f"  Test samples : {cnn_metrics['total_samples']}")

# ── Save model ─────────────────────────────────────────────────
torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : cnn_metrics
}, f"{MODELS}/exp3_cnn_full.pt")
print(f"✓ CNN model saved")

# ── Update tracker ─────────────────────────────────────────────
with open(f"{RESULTS}/experiment_tracker.json", "r") as f:
    tracker = json.load(f)

tracker['experiments']['exp3_full_upperbound']['status']      = "complete"
tracker['experiments']['exp3_full_upperbound']['nlp_metrics'] = nlp_metrics
tracker['experiments']['exp3_full_upperbound']['cnn_metrics'] = cnn_metrics
tracker['last_updated'] = str(datetime.now().date())

with open(f"{RESULTS}/experiment_tracker.json", "w") as f:
    json.dump(tracker, f, indent=2)

# ── Plot curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cnn_train_losses, color='#534AB7',
             linewidth=2, marker='o', markersize=3)
axes[0].set_title('CNN Loss — Full Dataset')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(cnn_train_accs, color='#BA7517',
             linewidth=2, marker='o', markersize=3)
axes[1].set_title('CNN Accuracy — Full Dataset')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS}/day5_cnn_curves.png",
            dpi=150, bbox_inches='tight')
plt.show()

# ── Save day summary ───────────────────────────────────────────
summary = {
    "day"        : 5,
    "experiment" : "exp3_full_upperbound",
    "train_size" : len(train_df),
    "test_size"  : len(test_df),
    "nlp_metrics": nlp_metrics,
    "cnn_metrics": cnn_metrics,
    "status"     : "Day 5 complete"
}

with open(f"{RESULTS}/day5_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 55)
print("DAY 5 COMPLETE ✓")
print("=" * 55)
print(f"  NLP Full — Accuracy : {nlp_metrics['accuracy']}%")
print(f"  NLP Full — Top-5    : {nlp_metrics['top5_accuracy']}%")
print(f"  CNN Full — Accuracy : {cnn_metrics['accuracy']}%")
print(f"  CNN Full — Top-5    : {cnn_metrics['top5_accuracy']}%")
print(f"\n  UPPER BOUND numbers confirmed ✓")
print(f"  Next → Day 6: HAM10000 on Kaggle")

Building CNN datasets...
✓ CNN DataLoaders ready
  Train images  : 29049
  Test images   : 7263
  Train batches : 908
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 127MB/s]



Training CNN — 15 epochs
  Images  : 29049
  Classes : 1199
-------------------------------------------------------


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

BASE = "/content/drive/MyDrive/rare_disease_project"

print("Zipping images folder...")
print("This will take 5-10 minutes...")

shutil.make_archive(
    f"{BASE}/data/zebramap_images",  # output zip name
    'zip',                            # format
    f"{BASE}/data/zebramap",         # folder to zip
    "images"                          # subfolder to zip
)

size = os.path.getsize(
    f"{BASE}/data/zebramap_images.zip") / (1024*1024*1024)
print(f"✓ ZIP created: zebramap_images.zip ({size:.2f} GB)")

Mounted at /content/drive
Zipping images folder...
This will take 5-10 minutes...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
from datetime import datetime

BASE      = "/content/drive/MyDrive/rare_disease_project"
IMAGE_SRC = f"{BASE}/data/zebramap/images"
ZIP_OUT   = f"{BASE}/data/zebramap_images.zip"

# Count files first
print("Counting files...")
total_files = 0
for root, dirs, files in os.walk(IMAGE_SRC):
    total_files += len([f for f in files
                        if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f"Total images to zip: {total_files}")

# Zip with lowest compression (speed priority)
print(f"\nStarting ZIP at {datetime.now().strftime('%H:%M:%S')}")
print("Progress will show every 500 files...")

count = 0
with zipfile.ZipFile(ZIP_OUT, 'w',
                     zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(IMAGE_SRC):
        for file in files:
            if file.lower().endswith(('.jpg','.jpeg','.png')):
                full_path = os.path.join(root, file)
                arc_name  = os.path.relpath(full_path, IMAGE_SRC)
                zf.write(full_path, arc_name)
                count += 1
                if count % 500 == 0:
                    print(f"  {count}/{total_files} files zipped "
                          f"({count/total_files*100:.1f}%)")

size = os.path.getsize(ZIP_OUT) / (1024*1024*1024)
print(f"\n✓ ZIP done at {datetime.now().strftime('%H:%M:%S')}")
print(f"✓ Size: {size:.2f} GB")
print(f"✓ Location: {ZIP_OUT}")

Mounted at /content/drive
Counting files...
Total images to zip: 94384

Starting ZIP at 06:54:17
Progress will show every 500 files...
  500/94384 files zipped (0.5%)
  1000/94384 files zipped (1.1%)
  1500/94384 files zipped (1.6%)
  2000/94384 files zipped (2.1%)
  2500/94384 files zipped (2.6%)
  3000/94384 files zipped (3.2%)
  3500/94384 files zipped (3.7%)
  4000/94384 files zipped (4.2%)
  4500/94384 files zipped (4.8%)
  5000/94384 files zipped (5.3%)
  5500/94384 files zipped (5.8%)
  6000/94384 files zipped (6.4%)
  6500/94384 files zipped (6.9%)
  7000/94384 files zipped (7.4%)
  7500/94384 files zipped (7.9%)
  8000/94384 files zipped (8.5%)
  8500/94384 files zipped (9.0%)
  9000/94384 files zipped (9.5%)
  9500/94384 files zipped (10.1%)
  10000/94384 files zipped (10.6%)
  10500/94384 files zipped (11.1%)
  11000/94384 files zipped (11.7%)
  11500/94384 files zipped (12.2%)
  12000/94384 files zipped (12.7%)
  12500/94384 files zipped (13.2%)
  13000/94384 files zipped (